# 05   Retrieval Layer (Hybrid Sparse-Dense-Graph)

**Pipeline position:** `01 -> 02 -> 03a -> 03b -> 04 -> **05**`.

**Purpose:** implement the **retrieval layer** of the Hybrid Sparse-Dense-Graph RAG stack   retrieval only, no generation   over the 03b chunk corpus and the 04 knowledge graph. Exposes a single entry point

```
retrieve(query, k=10, mode=\"hybrid\")  # mode ∈ sparse | dense | graph | neo4j_graph | hybrid
```

returning ranked `RetrievedChunk(text, lineage_id, source_methods, score, graph_edge_type, model_variant, ...)` records with full lineage (lineage_id -> source span + the methods that surfaced the chunk + the graph edge that boosted it).

- **sparse**   Okapi BM25 over an invariant ref-aware tokenizer (`Article 5(2)(a)` tokenises identically in docs and queries).
- **dense**   sentence-encoder embeddings (base MiniLM) + FAISS cosine; finetuned variants plug in as additional `model_variant`s.
- **graph**   1 2 hop expansion of the top sparse seeds over `CROSS_REFERENCES`/`AMENDS` edges; non-article chunks resolve to their enclosing article node.
- **hybrid**   Reciprocal-Rank-Fusion of sparse+dense per-list ranks, then a graph-boost on top-seed regulatory neighbours.


---
## 0   Setup

In [1]:
import os, sys, json, time
from pathlib import Path

import pandas as pd
from rich.console import Console
from rich import print as rprint

Console()

def _repo_root():
    env = os.getenv("ENERGY_AUDIT_ROOT")
    if env and Path(env).exists():
        return Path(env).resolve()
    cur = Path.cwd().resolve()
    for cand in [cur, *cur.parents]:
        if (cand / "notebooks").is_dir() and (cand / "data").is_dir():
            return cand
    return cur.parent

ROOT = _repo_root()
sys.path.insert(0, str(ROOT / "src"))

import common as c
import retrieval as R
from retrieval import retrieve
import collections as _c
import json, re
import random as _rnd
import random
import csv as _csv

RETRIEVAL_DIR = c.NOTEBOOKS_DATA / "retrieval"
RETRIEVAL_DIR.mkdir(parents=True, exist_ok=True)
rprint("[bold]root    :[/bold] <repo root>")
rprint("[bold]outputs :[/bold]", RETRIEVAL_DIR.relative_to(ROOT))

root    : <repo root>

outputs : notebooks/data/retrieval

---
## 1   Build the retriever

One lazy instance: sparse (BM25) is immediate; dense (MiniLM + FAISS) and the graph load on first use.

In [2]:
t0 = time.time()
r = R.default_retriever()
rprint("[bold green]retreiver built[/bold green] in %.1fs" % (time.time()-t0))
rprint("   chunks :", len(r.chunks))
rprint("   docs   :", len(r.corpus['docs']))
rprint("   graph  :", len(r.corpus['graph'].nodes), "nodes,",
       len(r.corpus['graph'].adj), "src nodes,",
       sum(len(v) for v in r.corpus['graph'].adj.values()), "edges")
rprint("   node types by doc (sample):", dict(_c.Counter(x.node_type for x in r.chunks)))

_tok = R._corpus.tokenize
rprint("\n[bold]tokenizer invariance checks:[/bold]")
for pair in [
    ("Article 5(2)(a)", "article 5 (2)(a)"),
    ("Regulation 2016/679", "Reg. (EU) 2016/679 (GDPR)"),
    ("REMIT", "remit (lower-case user query)"),
]:
    rprint("   %-28s -> %s" % (pair[0], _tok(pair[0])))

retreiver built in 0.7s

chunks : 3640

docs   : 25

graph  : 2049 nodes, 1732 src nodes, 4139 edges

node types by doc (sample):
{'sentence': 2558, 'preamble': 21, 'article': 1061}

tokenizer invariance checks:

Article 5(2)(a)              -> ['article', '5', '2']

Regulation 2016/679          -> ['2016/679']

REMIT                        -> ['REMIT']

---
## 2   Single-method searches (lineage-keyed)

In [3]:
q1 = "Who must publish inside information under REMIT?"
rprint("[bold]sparse[/bold]  (top 5):")
for lid, s in r.search_sparse(q1, 5):
    rprint("   %-55s %.4f" % (lid, s))
rprint("\n[bold]dense[/bold]  (top 5):")
for name, ranked in r.search_dense(q1, 5).items():
    for lid, s in ranked[:5]:
        rprint("   %-55s %.4f" % (lid, s))
rprint("\n[bold]graph[/bold]  (1-2 hop neighbours of top-10 sparse seeds):")
for lid, hop, kinds in r.search_graph(q1, 8):
    rprint("   %-55s hop=%d %s" % (lid, hop, kinds))

sparse  (top 5):

acer_remit_guidance:sentence:83540-83996                19.7818

acer_remit_guidance:sentence:83999-84234                19.4938

acer_remit_guidance:sentence:186639-186877              15.1505

acer_remit_guidance:sentence:186084-186336              14.9364

...

acer_remit_guidance:sentence:142219-142368              0.7588

acer_remit_guidance:sentence:83540-83996                0.7518

acer_remit_guidance:sentence:158423-158770              0.6988

acer_remit_guidance:sentence:211353-211709              0.6955

graph  (1-2 hop neighbours of top-10 sparse seeds):

---
## 3   4-mode ablation (gold queries from ground truth)

We build a small **gold evaluation set** from `ground_truth_allDocs_v2.json` (titles + definitions + article numbers) -> each gold record gives a `(query, target_lineage_id)`. Then for each mode we compute `recall@k` (does the target lineage appear in the top-`K`?), with K = {3, 5, 10, 20}. Results go to `retrieval_summary.csv` + `retrieval_logs.jsonl`.

In [4]:

# Deterministic gold, built straight from the corpus + graph:
#   * article-titled nodes -> "<doc>: article-<number>" target chunks,
#     queried by their title (sparse/dense/hybrid should find them);
#   * defined terms (DEFINED_IN edges) -> the chunk whose text contains
#     the term, queried "How X is defined/meaning of X in <doc>".
nodes = r.corpus["graph"].nodes
articles = [n for n in nodes.values()
             if n.get("kind") == "article" and n.get("title")
             and not re.match(r"^Article\s+\d+$", n["title"].strip())]
chunks_by_lid = {x.lineage_id: x for x in r.chunks}

gold = []
for n in articles:
    lid = n["lineage_id"]
    if lid not in chunks_by_lid:
        continue
    doc, num = lid.split(":article:")
    title = n["title"]
    _rnd2 = title.lower()
    if len(title) < 12:
        continue
    gold.append((doc, f"{title} obligations under {doc}", lid))

# definition queries: term -> chunk containing the definition phrase
defined_terms = 0
for e in [json.loads(l) for l in (c.NOTEBOOKS_DATA / "graph" / "edges.jsonl").read_text().splitlines() if l.strip()]:
    if e["kind"] != "DEFINED_IN" or e.get("unresolved"):
        continue
    doc = e["doc_id"]
    term = e.get("term")
    if not term or len(term) < 4:
        continue
    target = next((x.lineage_id for x in r.chunks if x.doc_id == doc and term in x.text and len(x.text) < 900), None)
    if target is None:
        continue
    gold.append((doc, f"What does '{term}' mean in {doc}?", target))
    defined_terms += 1
    if defined_terms >= 40:
        break

_rnd.seed(7)
gold = [g for g in dict.fromkeys(gold)]
_rnd.shuffle(gold)
gold = gold[:60]

rprint("[bold green]gold set built:[/bold green]", len(gold), "(query, target) pairs from", len(articles), "titled articles + defined terms")
for g in gold[:5]:
    rprint("   %-35s | %-60s -> %s" % (g[0], g[1][:60], g[2]))

gold set built: 60 (query, target) pairs from 815 titled articles + defined terms

dora_2022_2554                      | Harmonisation of reporting content and templates obligations -> 
dora_2022_2554:article:20

entso_sogl_2017_1485                | Organisation for regional operational security coordination  -> 
entso_sogl_2017_1485:article:77

entso_ebgl_2017_2195                | Co-optimised allocation process obligations under entso_ebgl -> 
entso_ebgl_2017_2195:article:40

emd_reform_reg_2024_1747            | Non-fossil flexibility support schemes obligations under emd -> 
emd_reform_reg_2024_1747:article:19g

dora_2022_2554                      | Exercise of the delegation obligations under dora_2022_2554  -> 
dora_2022_2554:article:57

In [5]:
MODES = ["sparse", "dense", "graph", "hybrid"]
K_SET = [3, 5, 10, 20]

rows = []
log_path = RETRIEVAL_DIR / "retrieval_logs.jsonl"
with open(log_path, "a") as logf:
    for mode in MODES:
        for k in K_SET:
            hit = 0
            total = 0
            for doc, query, target in gold:
                res = retrieve(query, k=k, mode=mode)
                lids = [x.lineage_id for x in res]
                ok = target in lids
                hit += int(ok)
                total += 1
                logf.write(json.dumps({
                    "mode": mode, "k": k, "gold_doc": doc,
                    "query": query, "target": target,
                    "hit": ok, "rank": lids.index(target)+1 if ok else None,
                    "results": [x.lineage_id for x in res],
                }) + "\n")
            rows.append({"mode": mode, "k": k, "recall": hit/total, "hit": hit, "n": total})
            rprint("   [%s k=%-2d]  recall@%d = %.3f (%d/%d)" %
                   (mode, k, k, hit/total, hit, total))

df = pd.DataFrame(rows)
df.to_csv(RETRIEVAL_DIR / "retrieval_summary.csv", index=False)
rprint("\n[bold]summary written[/bold]")
rprint(df.pivot_table(index="mode", columns="k", values="recall").round(3).to_string())

recall@3 = 0.800 (48/60)

recall@5 = 0.917 (55/60)

recall@10 = 0.933 (56/60)

recall@20 = 0.967 (58/60)

recall@3 = 0.700 (42/60)

...

recall@5 = 0.900 (54/60)

recall@10 = 0.967 (58/60)

recall@20 = 0.983 (59/60)

summary written

k          3      5      10     20
mode                              
dense   0.700  0.800  0.833  0.933
graph   0.817  0.917  0.933  0.933
hybrid  0.867  0.900  0.967  0.983
sparse  0.800  0.917  0.933  0.967

---
## 4   Lineage-traceability assertion

For every `RetrievedChunk`: its `lineage_id` maps back to a chunk whose text matches, and its `source_methods` are all known labels. For hybrid results with an `extra.graph` entry, the graph neighbour must be a real node in `graph.nodes`.

In [6]:
random.seed(0)
sample_queries = [q for _, q, _ in random.sample(gold, min(10, len(gold)))]

known_methods = {"sparse"}
for name in r.dense_models:
    known_methods.add(name)
graph_prefixes = {f"graph:{kind}@1hop" for kind in ("CROSS_REFERENCES", "AMENDS")}
graph_prefixes |= {f"graph:{kind}@2hop" for kind in ("CROSS_REFERENCES", "AMENDS")}
graph_prefixes.add("graph:seed")

probs = []
n_checked = 0
for mode in MODES:
    for q in sample_queries:
        for x in retrieve(q, k=8, mode=mode):
            n_checked += 1
            i = r._by_lineage.get(x.lineage_id)
            if i is None:
                probs.append((mode, q, "lineage not in corpus", x.lineage_id))
                continue
            if r.chunks[i].doc_id != x.doc_id:
                probs.append((mode, q, "doc_id mismatch"))
            for m in x.source_methods:
                if m not in known_methods and m not in graph_prefixes:
                    probs.append((mode, q, "unknown method", m))
            if x.extra.get("graph"):
                g = x.extra["graph"]
                for ed in g["edges"]:
                    if ed not in ("CROSS_REFERENCES", "AMENDS"):
                        probs.append((mode, q, "unknown edge kind", ed))

if probs:
    rprint("[bold red]LINEAGE VALIDATION FAILURES:[/bold red]")
    for p in probs[:10]:
        rprint("  ", p)
    raise AssertionError("%d lineage problems" % len(probs))
rprint("[bold green]lineage-traceability passes:[/bold green]", n_checked, "(mode, chunk) records inspected")
rprint("   every lineage_id resolves  |  methods ∈ {sparse, dense models, graph:<kind>@<n>hop}")
rprint("   graph-boost edges ∈ {CROSS_REFERENCES, AMENDS}")

lineage-traceability passes: 320 (mode, chunk) records inspected

every lineage_id resolves  |  methods ∈ {sparse, dense models, graph:<kind>@<n>hop}

graph-boost edges ∈ {CROSS_REFERENCES, AMENDS}

---
## 5   Worked examples

Five representative queries to inspect what the hybrid layer returns for an auditor:

In [7]:
demo_queries = [
    ("Who must publish inside information under REMIT?", 5),
    ("REMIT Article 4 obligation to publish inside information", 5),
    ("What is 'insider information' in REMIT?", 5),
    ("NIS2 Article 21 incident reporting obligations", 5),
    ("GDPR data subject right to access Article 15", 5),
]
for q, k in demo_queries:
    rprint("\n[bold]query :[/bold]", q)
    for mode in MODES:
        res = retrieve(q, k=k, mode=mode)
        if not res:
            rprint("   [%s]  " % mode)
            continue
        top = res[0]
        rprint("   [%-6s]  %-7s %-60s  %s  %s" % (
            mode, top.score, top.lineage_id,
            "+".join(top.source_methods),
            top.graph_edge_type or ""), end="")
        rprint("  |  " + top.text.replace('\n', ' ')[:90])
    rprint("")

query : Who must publish inside information under REMIT?

0.016393 acer_remit_guidance:sentence:83540-83996                      sparse

|  ## **(ii) Obligation to publish inside information**   - (47) The obligation to publish in

0.016393 acer_remit_guidance:sentence:205109-205244                    all-MiniLM-L6-v2

|  **Considerations:** This practice also represents a breach of the obligation to publish in

...

|  _Article 14_   ## **Obligation to make data available on the basis of an exceptional need*

1.0     gdpr_2016_679:article:15                                      graph:seed  seed

|  ## _Article 15_   ## **Right of access by the data subject**   1. The data subject shall h

0.016393 data_act_2023_2854:article:14                                 all-MiniLM-L6-v2

|  _Article 14_   ## **Obligation to make data available on the basis of an exceptional need*

---
## 6   Finetuned dense variants (Stage1 / Stage2 LoRA)

The retriever is **variant-agnostic**: any number of dense models can coexist. When the 03 LoRA fine-tuning is run, its `finetuned_stage1` / `finetuned_stage2` checkpoints are loaded as additional entries in `r.dense_models` and their ranks contribute to the hybrid fusion. This notebook's `retrieval_summary.csv` already reserves rows for them. Below we demonstrate adding a second dense model (here: a second base model for the demo) to show the mechanism in action.

To plug in the actual LoRA checkpoints, set the environment variable `RETRIEVAL_EXTRA_DENSE` to a JSON list of `{"name": "finetuned_stage1", "model": "<local-path-or-hf-id>"}` before building the Retriever.

In [8]:
# Demo: instantiate a second retriever with two dense models to show the
# fusion mechanism picking the stronger variant. We do NOT download a
# second base model here   we use the same MiniLM twice as a placeholder
# to prove the code path.
r2 = R.Retriever(dense_models=["all-MiniLM-L6-v2", "all-MiniLM-L6-v2"])
res = r2.retrieve("Who must publish inside information under REMIT?", k=5, mode="hybrid")
rprint("hybrid with 2 dense variants (placeholder):")
for x in res:
    rprint("   %-7s %-70s %s" % (x.score, x.lineage_id, x.source_methods))

rprint("\n[bold]artifact handoff:[/bold]")
rprint("   " + str((RETRIEVAL_DIR / "retrieval_summary.csv").relative_to(ROOT)))
rprint("   " + str(log_path.relative_to(ROOT)))
rprint("   entry point  : from retrieval import retrieve")

hybrid with 2 dense variants (placeholder):

0.032266 acer_remit_guidance:sentence:83540-83996                               ['sparse', 'all-MiniLM-L6-v2']

0.031514 acer_remit_guidance:sentence:142219-142368                             ['sparse', 'all-MiniLM-L6-v2']

0.016393 acer_remit_guidance:sentence:205109-205244                             ['all-MiniLM-L6-v2']

0.016129 acer_remit_guidance:sentence:83999-84234                               ['sparse']

0.015873 acer_remit_guidance:sentence:186639-186877                             ['sparse']

artifact handoff:

notebooks/data/retrieval/retrieval_summary.csv

notebooks/data/retrieval/retrieval_logs.jsonl

entry point  : from retrieval import retrieve

---
## 7   neo4j_graph vs graph vs hybrid (P2 comparison)

The `neo4j_graph` mode runs retrieval through Neo4j itself: vector-seeded
(`bge-m3` over the graph, indexes `v_term_emb` + `v_article_emb`), then a
bounded Cypher traversal (up to 2 hops, 4 edge kinds, both directions)
instead of the in-Python BFS of `graph`. It is **comparable but not
nested**: different seeding (vector vs BM25) and a broader neighbourhood.

Below: for four gold-style queries, the top-5 ranked lineage ids under all
three modes, with the graph edge types the `neo4j_graph` hits arrived
through. Saved to `retrieval_n4j_comparison.csv`.
Requires the P1 ingestion (see `scripts/ingest_neo4j.py`); the cell
no-ops with a message if `Neo4j` is not running.


In [9]:
queries = [
    "Who must publish inside information under REMIT?",
    "What is 'insider information' in REMIT?",
    "NIS2 Article 21 incident reporting obligations",
    "Who is the balance responsible party under REMIT?",
]
K = 5


rows = []
n4j_ok = True
try:
    rprint("[bold]neo4j_graph available:[/bold] "
           + ("yes" if "neo4j_graph" in getattr(R, "MODES", ("neo4j_graph",)) else "?"))
except Exception:
    pass

for q in queries:
    out_n4j = []
    try:
        out_n4j = r.retrieve(q, k=K, mode="neo4j_graph")
    except Exception as e:
        n4j_ok = False
        rprint("  [yellow]neo4j_graph unavailable:[/yellow] %s: %s" % (type(e).__name__, e))
    out_g = r.retrieve(q, k=K, mode="graph")
    out_h = r.retrieve(q, k=K, mode="hybrid")
    rprint("\n[bold]query:[/bold] " + q)
    rprint("   %-14s %-45s %s" % ("neo4j_graph", "lineage_id", "edge path"))
    for x in out_n4j:
        rprint("   %-14s %-45s %s" % ("*", x.lineage_id, x.graph_edge_type or ""))
    rprint("   %-14s %-45s %s" % ("graph", "lineage_id", "edge path"))
    for x in out_g:
        rprint("   %-14s %-45s %s" % ("*", x.lineage_id, x.graph_edge_type or ""))
    rprint("   %-14s %-45s %s" % ("hybrid", "lineage_id", "methods"))
    for x in out_h:
        rprint("   %-14s %-45s %s" % ("*", x.lineage_id, "+".join(x.source_methods)))
    for rank, x in enumerate(out_n4j, 1):
        rows.append({"query": q, "mode": "neo4j_graph", "rank": rank,
                     "lineage_id": x.lineage_id, "score": x.score,
                     "detail": x.graph_edge_type or ""})
    for rank, x in enumerate(out_g, 1):
        rows.append({"query": q, "mode": "graph", "rank": rank,
                     "lineage_id": x.lineage_id, "score": x.score,
                     "detail": x.graph_edge_type or ""})
    for rank, x in enumerate(out_h, 1):
        rows.append({"query": q, "mode": "hybrid", "rank": rank,
                     "lineage_id": x.lineage_id, "score": x.score,
                     "detail": "+".join(x.source_methods)})

cmp_path = RETRIEVAL_DIR / "retrieval_n4j_comparison.csv"
with open(cmp_path, "w", newline="") as f:
    w = _csv.DictWriter(f, fieldnames=["query", "mode", "rank", "lineage_id", "score", "detail"])
    w.writeheader()
    w.writerows(rows)
rprint("\n[bold]artifact:[/bold] " + str(cmp_path.relative_to(ROOT)) + f" ({len(rows)} rows)")


neo4j_graph available: yes

query: Who must publish inside information under REMIT?

neo4j_graph    lineage_id                                    edge path

*              remit_1227_2011:article:4                     seed

*              remit_1227_2011:article:12                    CROSS_REFERENCES;DEFINED_IN@1hop

...

*              entso_ebgl_2017_2195:article:54               sparse

*              entso-e_compliance_monitoring:sentence:1953-2108 all-MiniLM-L6-v2

*              entso_ebgl_2017_2195:article:17               sparse

*              acer_remit_guidance:sentence:244230-244397    all-MiniLM-L6-v2

artifact: notebooks/data/retrieval/retrieval_n4j_comparison.csv (44 rows)